<a href="https://colab.research.google.com/github/minaGalil/Imperial-Capstone/blob/main/Capstone_Week_5_Calling_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

np.random.seed(42)

# =========================================================
# LOAD YOUR WEEK 1–4 DATA (reuse same arrays you already have)
# =========================================================

# 👉 keep your existing week1_inputs, week2_inputs, week3_inputs, week4_inputs
# 👉 same for outputs
# (no need to repeat here — reuse from Week 4 code)


# =========================================================
# WEEK DATA
# =========================================================

week1_inputs = [
    [0.761024, 0.713000],
    [0.732637, 0.906564],
    [0.522581, 0.591593, 0.350176],
    [0.564020, 0.473834, 0.390972, 0.258427],
    [0.196777, 0.892275, 0.855813, 0.891829],
    [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
    [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
    [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
]

week1_outputs = [
    -7.565331332744927e-18,
    0.5530064492925906,
    -0.03333218343962805,
    -4.83054096204485,
    1257.680268889983,
    -0.8072367077314392,
    1.2394822144938658,
    9.5430069620611,
]

week2_inputs = [
    [0.751825, 0.811946],
    [0.980817, 0.626715],
    [0.996446, 0.737055, 0.850924],
    [0.404477, 0.413254, 0.303108, 0.434359],
    [0.521278, 0.505138, 0.985473, 0.994545],
    [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
    [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
    [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
]

week2_outputs = [
    2.495129202697582e-35,
    0.06646691679004207,
    -0.04126707799435369,
    -0.7158150747637886,
    1769.2992166742467,
    -0.721361811601727,
    1.493591747104962,
    9.7741312776374,
]

week3_inputs = [
    [0.702630, 0.955980],
    [0.700329, 1.000000],
    [0.977206, 0.330727, 0.000017],
    [0.393932, 0.393230, 0.377559, 0.426151],
    [0.449325, 0.603715, 1.000000, 1.000000],
    [0.941155, 0.654101, 0.079248, 0.961137, 0.229746],
    [0.153100, 0.236737, 0.190018, 0.000000, 0.365128, 0.744155],
    [0.103762, 0.017606, 0.204751, 0.194101, 0.745367, 0.674182, 0.125362, 0.044024],
]

week3_outputs = [
    -8.189634555426656e-79,
    0.5980717829505922,
    -0.12802055717541724,
    0.2735224699683667,
    2027.715300336871,
    -1.6878746934171067,
    1.120167075371798,
    9.8898511444579,
]

week4_inputs = [
    [0.000410-0.596183],
    [0.005570-0.993176],
    [0.410156-0.996643-0.021209],
    [0.407898-0.389916-0.369687-0.424652],
    [0.860077-0.983344-0.984180-0.991847],
    [0.241982-0.288951-0.395099-0.781426-0.007278],
    [0.080199-0.220230-0.061424-0.211450-0.460647-0.550577],
    [0.000000-0.129759-0.000000-0.243975-1.000000-1.000000-0.000000-0.000000],
]
"""
week4_outputs = [
    -8.189634555426656e-79,
    0.5980717829505922,
    -0.12802055717541724,
    0.2735224699683667,
    2027.715300336871,
    -1.6878746934171067,
    1.120167075371798,
    9.8898511444579,
]
"""
# =========================================================
# HELPER FUNCTIONS
# =========================================================

def relu(x): return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)

def mlp_gradient(x_scaled, mlp):
    coefs = mlp.coefs_
    intercepts = mlp.intercepts_

    activations = [x_scaled.reshape(1, -1)]
    pre = []

    a = activations[0]

    for i in range(len(coefs)-1):
        z = a @ coefs[i] + intercepts[i]
        pre.append(z)
        a = relu(z)
        activations.append(a)

    z_out = a @ coefs[-1] + intercepts[-1]
    pre.append(z_out)

    delta = np.ones_like(z_out) @ coefs[-1].T

    for i in reversed(range(len(coefs)-1)):
        delta *= relu_grad(pre[i])
        if i == 0:
            grad = delta @ coefs[i].T
        else:
            delta = delta @ coefs[i].T

    return grad.ravel()

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

def feature_importance(X, scaler, mlp):
    grads = []
    for x in X:
        xs = scaler.transform(x.reshape(1,-1)).ravel()
        g = mlp_gradient(xs, mlp)
        grads.append(np.abs(g))
    grads = np.array(grads)
    imp = grads.mean(axis=0)
    return imp / imp.sum()

# =========================================================
# MAIN LOOP
# =========================================================

results = []

for i in range(8):

    print(f"\nFunction {i+1}")

    X = np.load(f"function{i+1}_inputs.npy")
    y = np.load(f"function{i+1}_outputs.npy")

    # Append Week 1–4 (reuse your logic here)

    best_idx = np.argmax(y)
    best_input = X[best_idx]

    dim = X.shape[1]

    # =====================================================
    # MODELS
    # =====================================================

    # GP
    gp = GaussianProcessRegressor(normalize_y=True)
    gp.fit(X, y)

    # SVM
    threshold = np.quantile(y, 0.7)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) > 1:
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True))
        ])
        svm.fit(X, labels)
        use_svm = True
    else:
        use_svm = False

    # Deep NN (Week 5 upgrade)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    nn = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),  # deeper network
        activation="relu",
        alpha=0.0005,  # regularization
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    )
    nn.fit(Xs, y)

    # =====================================================
    # FEATURE IMPORTANCE
    # =====================================================
    importance = feature_importance(X, scaler, nn)

    # =====================================================
    # MULTI-START GRADIENT ASCENT
    # =====================================================
    candidates = []

    top_idx = np.argsort(y)[-5:]

    for idx in top_idx:
        x = X[idx].copy()

        for _ in range(30):
            xs = scaler.transform(x.reshape(1,-1)).ravel()
            grad = mlp_gradient(xs, nn)

            # scale gradient by feature importance
            grad = grad * importance

            norm = np.linalg.norm(grad)
            if norm > 0:
                grad = grad / norm

            x = x + 0.05 * grad
            x = np.clip(x, 0, 1)

        candidates.append(x)

    candidates = np.array(candidates)

    # =====================================================
    # GLOBAL + LOCAL SEARCH
    # =====================================================
    global_cand = np.random.uniform(0,1,(8000,dim))
    local_cand = np.clip(best_input + np.random.normal(0,0.05,(2000,dim)),0,1)

    all_candidates = np.vstack([global_cand, local_cand, candidates])

    # =====================================================
    # SCORING
    # =====================================================
    mu, sigma = gp.predict(all_candidates, return_std=True)
    ucb = mu + 2*sigma

    nn_pred = nn.predict(scaler.transform(all_candidates))

    if use_svm:
        svm_prob = svm.predict_proba(all_candidates)[:,1]
    else:
        svm_prob = np.ones(len(all_candidates))

    # NEW: confidence from NN gradient magnitude
    grad_strength = np.linalg.norm(
        [mlp_gradient(scaler.transform(x.reshape(1,-1)).ravel(), nn)
         for x in all_candidates], axis=1
    )

    # FINAL SCORE (Week 5)
    score = (
        0.35 * ucb +
        0.30 * nn_pred +
        0.20 * svm_prob +
        0.15 * grad_strength
    )

    best_idx = np.argmax(score)
    new_point = all_candidates[best_idx]

    query = "-".join([f"{v:.6f}" for v in new_point])

    print("Submission:", query)

    results.append({
        "Function": i+1,
        "Query": query
    })

pd.DataFrame(results).to_csv("Week5_Submissions.csv", index=False)

print("\nWeek 5 Done ✅")


Function 1
Submission: 0.012715-0.992415

Function 2
Submission: 0.026867-0.999598

Function 3
Submission: 0.955194-0.008179-0.700062

Function 4
Submission: 0.425212-0.377719-0.427527-0.420346

Function 5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Submission: 0.479838-0.890814-1.000000-1.000000

Function 6
Submission: 0.137088-0.268975-0.283122-0.986752-0.045377

Function 7
Submission: 0.000000-0.450181-0.211028-0.000000-0.353842-1.000000

Function 8
Submission: 0.000000-0.000000-0.301242-0.216500-0.000000-1.000000-0.000000-1.000000

Week 5 Done ✅
